In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

catalog_name = 'ecommerce'

### Brands

In [0]:
df_bronze = spark.table(f'{catalog_name}.bronze.brz_brands')

In [0]:
display(df_bronze.limit(10))  

In [0]:
# trim the brand_name column

df_silver = df_bronze.withColumn('brand_name',trim(col('brand_name')))
display(df_silver.limit(5))  

In [0]:
# remove special characters from brand_code

df_silver = df_silver.withColumn('brand_code',regexp_replace(col('brand_code'),r'[^A-Za-z0-9]',''))
display(df_silver.limit(10))  

In [0]:
# identifying duplicate category

df_silver.select(col('category_code')).distinct().show()

In [0]:
# category_code column clean up

anomalies = {
    'BOOKS':'BKS',
    'GROCERY':'GRCY',
    'TOYS':'TOY'
}

df_silver = df_silver.replace(anomalies, subset='category_code')
df_silver.select(col('category_code')).distinct().show()

In [0]:
# create the final table in the silver layer

df_silver.write.format('delta').mode('overwrite').option('mergeSchema','true').saveAsTable(f'{catalog_name}.silver.slv_brands')

### Category

In [0]:
df_bronze = spark.table(f'{catalog_name}.bronze.brz_category')

display(df_bronze.limit(5))

In [0]:
# identify duplicates based on category code

df_bronze.groupBy(col('category_code')).count().where(col('count')>1).show()


In [0]:
# remove the duplicate categoy_code

df_silver = df_bronze.dropDuplicates(['category_code'])

df_silver.groupBy(col('category_code')).count().show()

In [0]:
# change category_code to upper case

df_silver = df_silver.withColumn('category_code', upper(col('category_code')))

df_silver.select(col('category_code')).distinct().show()

In [0]:
# create the final table in the silver layer

df_silver.write.format('delta').mode('overwrite').option('mergeSchema','true').saveAsTable(f'{catalog_name}.silver.slv_category')

### Products

In [0]:
df_bronze = spark.read.table(f'{catalog_name}.bronze.brz_products')

display(df_bronze.limit(5))

In [0]:
# row and column counts

row_count, col_count = df_bronze.count(), len(df_bronze.columns)

print(f'Row Count is {row_count} and Column Count is {col_count}.')

In [0]:
# replace 'g' in weight_grams column with ''

df_silver = df_bronze.withColumn('weight_grams', regexp_replace(col('weight_grams'),'g','').cast(IntegerType()))

df_silver.select(col('weight_grams')).show(5, truncate=False)

In [0]:
# replace , in length_cm column

df_silver = df_bronze.withColumn('length_cm', regexp_replace(col('length_cm'),',','.').cast(FloatType()))

df_silver.select(col('length_cm')).show(5, truncate=False)

In [0]:
# change category_code to upper case

df_silver = df_silver.withColumn('category_code', upper(col('category_code'))).withColumn('brand_code',upper(col('brand_code')))

df_silver.select(col('category_code')).distinct().show()
df_silver.select(col('brand_code')).distinct().show()

In [0]:
# fix spelling erros

df_silver = df_silver.withColumn('material',
                     when(col('material')=='Coton','Cotton').
                     when(col('material')=='Alumium','Aluminium').when(col('material')=='Ruber','Rubber').otherwise(col('material')))

df_silver.select(col('material')).distinct().show()  

In [0]:
# identify negative rating_count

df_silver.where(col('rating_count')<0).select('rating_count').show(5)

In [0]:
# convert negative rating_count to positive

df_silver = df_silver.withColumn('rating_count',
                     when(col('rating_count').isNotNull(), abs(col('rating_count'))).otherwise(lit(0))
                     )

df_silver.where(col('rating_count')<0).count()                     

In [0]:
# create the final table in the silver layer

df_silver.write.format('delta').mode('overwrite').option('mergeSchema','true').saveAsTable(f'{catalog_name}.silver.slv_products')

### Customers

In [0]:
df_bronze = spark.read.table(f'{catalog_name}.bronze.brz_customer')

display(df_bronze.limit(5))

In [0]:
# identify null values in customer_id

df_bronze.where(col('customer_id').isNull()).count()

In [0]:
# drop null customer_id

df_silver = df_bronze.dropna(subset=['customer_id'])

df_silver.where(col('customer_id').isNull()).count()

In [0]:
# identify null values in phone

df_silver.where(col('phone').isNull()).count()

In [0]:
# fill the null phone

df_silver = df_silver.fillna('NA',subset=['phone'])

In [0]:
# create the final table in the silver layer

df_silver.write.format('delta').mode('overwrite').option('mergeSchema','true').saveAsTable(f'{catalog_name}.silver.slv_customers')

### Date

In [0]:
df_bronze = spark.read.table(f'{catalog_name}.bronze.brz_calendar')

display(df_bronze.limit(5))

In [0]:
# convert data type of date

df_silver = df_bronze.withColumn('date',to_date(col('date'),'dd-MM-yyyy'))

df_silver.printSchema()

In [0]:
# identify duplicate rows

df_silver.groupBy(col('date')).count().where(col('count')>1).show()

In [0]:
# remove duplicate rows

df_silver = df_silver.dropDuplicates(subset=['date'])

df_silver.groupBy(col('date')).count().where(col('count')>1).show()

In [0]:
# change the case of day_name

df_silver = df_silver.withColumn('day_name',initcap(col('day_name')))

df_silver.select(col('day_name')).distinct().show()

In [0]:
# convert negative week_of_year to positive

df_silver = df_silver.withColumn('week_of_year', abs(col('week_of_year')))

In [0]:
# enhance quarter and week_of_year column

df_silver = df_silver.withColumn('quarter',concat_ws('',concat(lit('Q'),col('quarter'),lit('-'),col('year'))))

df_silver = df_silver.withColumn('week_of_year',concat_ws('-',concat(lit('Week'),col('week_of_year'),lit('-'),col('year'))))

display(df_silver.limit(5))


In [0]:
# rename week_of_year

df_silver = df_silver.withColumnRenamed('week_of_year','week')

In [0]:
# create the final table in the silver layer

df_silver.write.format('delta').mode('overwrite').option('mergeSchema','true').saveAsTable(f'{catalog_name}.silver.slv_calendar')